This work is done under Kernal Python 3.10.6

pre-work preparation works:

In [ ]:
import requests
import pandas as pd

from pprint import pprint     
from scrapy import Selector

# Part 1

Scrap the list of events from the CIVICA Schedule page

1: request a website and check whether it works

In [ ]:
url = 'https://socialdatascience.network/index.html#schedule'
response = requests.get(url)
response

2: prepare for getting data

In [ ]:
# define sel
sel = Selector(text=response.text)

3: scrap the list of events from the CIVICA Schedule page

In [ ]:
# 3.1 scrap title
# through google inspect we know TITLE is at "h6"
titles = sel.css('h6.card-title ::text').getall()
# print it with panda (friendly for reading)
pd.Series(titles)

In [ ]:
# 3.2 scrap speaker and date
# through google inspect, we know SPEAKER and DATA are at "p"
speakers = []
dates = []

for i in sel.css('div.card-body p ::text').getall():
    i = i.lstrip()
    if i.startswith('Speaker:'):
        speaker_name = i.replace('Speaker: ', '')
        speakers.append(speaker_name)
    elif i.startswith('Date:'):
        date = i.replace('Date: ', '')
        dates.append(date)
    else:
        speakers.append('')
# print list of speakers
pd.Series(speakers)

In [ ]:
# print list of dates
pd.Series(dates)

In [ ]:
# 3.3 scrap links
# through google inspect, we know links are at "div.card-body", and we use href to get links.
links = sel.css('div.card-body ::attr(href)').getall()
#prink links
pprint(links)

4: **problems** encountered when scrapping links

4.1 if we use the code 
df = pd.DataFrame({'title': titles, 'speaker': speakers, 'date': dates, 'link': links})
df.to_csv('schedule.csv', index=False)
the **ValueError: All arrays must be of the same length** will occur

4.2 the file wont be generated because we have a 74 lines data of link, but there are 36 lines data for title, speaker and date. 

Thus we need to mannually find the valid links and re-code. to cancel **duplicate data**
We define the links data of **74** lines as duplicate_links

In [ ]:
links = sel.css('div.card-body ::attr(href)').getall()
duplicate_links=[]
print(links)
# write the first loop to eliminate excess links
for i in links:
    if i.startswith('fall') or i.startswith('spring') or i.startswith('sess') or i.startswith('launch') or i.startswith('polarisation'): #valid links
        duplicate_links.append(i)
links=list(set(duplicate_links))
#lets check
print(len(links))
pd.Series(links)

duplicate links= links repeated, links =links unprocessed, final links=links

However, we could find out in the output that the links are in **random orders**. 
To solve this, we define the final outputs as final_links.

In [ ]:
final_links = []
# we write a second for loop here to correct the order.
for item in duplicate_links:
    if item not in final_links:
        final_links.append(item)
pd.Series(final_links)
# "final_links",this is the final one we want to use in the following code

5. Save data into a csv file

In [ ]:
df = pd.DataFrame({'title': titles, 'speaker': speakers, 'date': dates, 'link': final_links})
df.to_csv('schedule.csv', index=False)

# Part2

1: we do preparation works, and define the URLs first

In [ ]:
# preparation works:
import requests
import re
from scrapy import Selector
# define URLs:
part2_urls = [] #first define an empty parameter
for i in final_links:
    generate_part2_urls = ('https://socialdatascience.network/'+ i)
    part2_urls.append(generate_part2_urls) # now we define the part2_urls
# (the above "generate_part2_urls" is simply combinations of the 
# common prefix: "https://socialdatascience.network/" + the 37 links from "final_links")

1.1 ok lets check if urls work, and yes it is.

In [ ]:
pd.Series(part2_urls)

1.2 now, time to cope with the selectors

In [ ]:
responses2 = []

# for loop
for i in part2_urls:
    response = requests.get(i)
    responses2.append(response.text)

# well another for loop, this time for sel2
sel2 = []
for html_content in responses2:
    selector = Selector(text=html_content)
    sel2.append(selector)

# does it respond? yes. [200]. perfect
response


2：OK now we have made the definations. Time to work on the procedure of scraping data from URLs

2.1: scrap the event_titles(the links to the events), this is a quite easy one

In [ ]:
event_titles = []
for i in final_links:
    ET = part2_urls = ('https://socialdatascience.network/'+ i)
    event_titles.append(ET)
pd.Series(event_titles)

2.2: scrap the talk_titils and event_speakers, and seperate them.

In [ ]:
# from google inspect we know the information we need are at "h4"
raw_data = []
for X in range (0,38):
    generate_raw_data=sel2[X].css('h4 ::text').getall(),
    raw_data.append(generate_raw_data) #now we have raw data, lets process on it
pprint(raw_data)
# or we could try pd.Series, both will work.
pd.Series(raw_data)

Honestly my first choice is to keep writing stuff like talk_titles=sel2[0].css('h4 ::text').getall(), talk_titles=sel2[1].css('h4 ::text').getall(), talk_titles=sel2[2].css('h4 ::text').getall() ... all the way to 38 though, when I copy and paste till [20], I realized that I could use a **for loop** with a **defined range** 1-38 with a parameter **X**.

2.3 now lets separate data

2.3.1 _I have demenstrated an example of how to filter the data I want as following code_

In [ ]:
random_sample = [(1, 2, 3, 4, 5, 6, 7), (8, 9, 10, 11, 12, 13, 14)]
new_list = [(t[1], t[3]) for t in random_sample]
print(new_list)

2.3.2 _and I can do exactly the same to "raw_data"_, and get filtered talk_titils and event_speakers

In [ ]:
talk_speakers = []

for j in raw_data:
    if len(j[0]) >= 2: # make sure there is data in the line
        talkspeaker = (j[0][1], j[0][3], j[0][5]) #this looks different because raw_data is a list of tuples, we need to first access[0], then filter[1].
        talk_speakers.append(talkspeaker)
    else:
        talk_speakers.append(()) #if its empty, let it be empty, so its still 38 lines, facilitas our last file-saving part

pd.Series(talk_speakers)


In [ ]:
talk_titles = []

for k in raw_data:
    if len(k[0]) >= 2:
        talktitle = (k[0][0], k[0][2], k[0][4], k[0][6])
        talk_titles.append(talktitle)
    else:
        talk_titles.append(())

pd.Series(talk_titles)

2.4 now scrap talk_description

In [ ]:
# from google inspect we know the information we need are at "div", specifically col-md-10. 
# Just keep trying different ones in the bracket of css() untill finding the most suitable one.
talk_description = []
for Y in range (0,38):
    talks=sel2[Y].css('div.col-md-10 p ::text').getall(),
    talk_description.append(talks)
pd.Series(talk_description)

38 lines of data, which proves we got the correct one.

3: Now we have all the data we want, lets create the CSV file called **agenda**

In [ ]:
df2 = pd.DataFrame({'event_title': event_titles, 'talk_title': talk_titles, 'talk_speakers': talk_speakers, 'talk_description': talk_description})
df2.to_csv('agenda.csv', index=False)